# 193. INFERCEPT：Agent 被工具或人工中断后，怎样保留 KV 并安全恢复？

> **面试问题：怎样区分 pause 与结束、用 snapshot 防止错误 KV 复用，并在显存预算下选择保留或重算？**

## 先给结论

高质量回答需要同时说明目标状态、可执行策略、状态版本、失败回滚和可复放评测。下面不用 Agent 框架或远程工具，而是用受控的内存模型把核心合同写出来；断言只证明教学实现的不变量，不能替代真实服务的隔离、审计、权限与压测。

## 一手资料

- [INFERCEPT](https://arxiv.org/abs/2402.01869)
- [PagedAttention / vLLM](https://arxiv.org/abs/2309.06180)
- [ReAct](https://arxiv.org/abs/2210.03629)

In [ ]:
notebook_contract = {"mode": "in-memory-demo", "oracle": "explicit-assertions", "production": "needs-isolation"}  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["mode"] == "in-memory-demo"  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["oracle"] == "explicit-assertions"  # 执行本行的状态、计算或校验逻辑。
assert "isolation" in notebook_contract["production"]  # 执行本行的状态、计算或校验逻辑。
assert len(notebook_contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 问题拆解：工具调用不是请求结束

Agent 等待工具或人工输入时，解码暂停但此前 prompt 与生成 token 的 KV 仍会在恢复时使用。若直接丢弃，恢复会重新 prefill 整段上下文。INFERCEPT 讨论的正是这类 interception；此处用 token 数和 KV block 标识模拟，而不假装实现 GPU cache。


In [ ]:
from dataclasses import dataclass, field  # 执行本行的状态、计算或校验逻辑。
@dataclass  # 执行本行的状态、计算或校验逻辑。
class AgentSession:  # 执行本行的状态、计算或校验逻辑。
    session_id: str  # 执行本行的状态、计算或校验逻辑。
    tokens: list  # 执行本行的状态、计算或校验逻辑。
    kv_blocks: list  # 执行本行的状态、计算或校验逻辑。
    version: int = 1  # 执行本行的状态、计算或校验逻辑。
    phase: str = "decoding"  # 执行本行的状态、计算或校验逻辑。
    intercept_kind: str | None = None  # 执行本行的状态、计算或校验逻辑。
session = AgentSession("s-1", [11, 12, 13], ["kv-0", "kv-1"])  # 执行本行的状态、计算或校验逻辑。
assert session.phase == "decoding"  # 执行本行的状态、计算或校验逻辑。
assert len(session.tokens) == 3  # 执行本行的状态、计算或校验逻辑。
assert len(session.kv_blocks) == 2  # 执行本行的状态、计算或校验逻辑。


## 2. 暂停合同：冻结版本并标记不可调度

暂停不是释放请求：它需要记录等待什么、保留哪些 KV block、在哪个版本冻结。真实调度器还要分别处理短工具调用和长人工等待，并在内存压力下有可解释的 eviction 策略。


In [ ]:
def pause(session, kind):  # 执行本行的状态、计算或校验逻辑。
    if session.phase != "decoding":  # 执行本行的状态、计算或校验逻辑。
        raise ValueError("只有正在解码的会话可以暂停")  # 执行本行的状态、计算或校验逻辑。
    session.phase = "paused"  # 执行本行的状态、计算或校验逻辑。
    session.intercept_kind = kind  # 执行本行的状态、计算或校验逻辑。
    return {"version": session.version, "kv": tuple(session.kv_blocks), "kind": kind}  # 执行本行的状态、计算或校验逻辑。
snapshot = pause(session, "tool")  # 执行本行的状态、计算或校验逻辑。
assert session.phase == "paused"  # 执行本行的状态、计算或校验逻辑。
assert snapshot["kv"] == ("kv-0", "kv-1")  # 执行本行的状态、计算或校验逻辑。
assert snapshot["kind"] == "tool"  # 执行本行的状态、计算或校验逻辑。


## 3. 恢复合同：匹配 snapshot 才能复用 KV

恢复时，工具结果、会话版本和保留的 KV 必须共同匹配；否则把旧上下文接到新请求上会造成跨请求污染。KV 命中路径的额外 prefill 成本为零，这里只计算逻辑 token 工作量，不代表 GPU 时延。


In [ ]:
def resume(session, snapshot, tool_tokens):  # 执行本行的状态、计算或校验逻辑。
    if session.phase != "paused" or snapshot["version"] != session.version:  # 执行本行的状态、计算或校验逻辑。
        raise ValueError("恢复快照与会话不兼容")  # 执行本行的状态、计算或校验逻辑。
    if tuple(session.kv_blocks) != snapshot["kv"]:  # 执行本行的状态、计算或校验逻辑。
        raise ValueError("KV 内容已变化，不能复用")  # 执行本行的状态、计算或校验逻辑。
    session.tokens.extend(tool_tokens)  # 执行本行的状态、计算或校验逻辑。
    session.phase = "decoding"  # 执行本行的状态、计算或校验逻辑。
    session.intercept_kind = None  # 执行本行的状态、计算或校验逻辑。
    return {"extra_prefill_tokens": 0, "total_tokens": len(session.tokens)}  # 执行本行的状态、计算或校验逻辑。
resumed = resume(session, snapshot, [99])  # 执行本行的状态、计算或校验逻辑。
assert resumed["extra_prefill_tokens"] == 0  # 执行本行的状态、计算或校验逻辑。
assert session.tokens[-1] == 99  # 执行本行的状态、计算或校验逻辑。
assert session.phase == "decoding"  # 执行本行的状态、计算或校验逻辑。


## 4. 对照路径：丢弃缓存会重复 prefill

为量化取舍，必须保留 discard baseline。丢弃后恢复不是错误，但它需要重新处理全部已知上下文；在真实服务中还会牺牲吞吐。不同任务的暂停时长和上下文长度决定是否值得保留。


In [ ]:
def discard_and_resume(tokens, tool_tokens):  # 执行本行的状态、计算或校验逻辑。
    rebuilt = list(tokens) + list(tool_tokens)  # 执行本行的状态、计算或校验逻辑。
    return {"extra_prefill_tokens": len(rebuilt), "rebuilt_tokens": rebuilt}  # 执行本行的状态、计算或校验逻辑。
discarded = discard_and_resume([11, 12, 13], [99])  # 执行本行的状态、计算或校验逻辑。
assert discarded["extra_prefill_tokens"] == 4  # 执行本行的状态、计算或校验逻辑。
assert discarded["rebuilt_tokens"] == [11, 12, 13, 99]  # 执行本行的状态、计算或校验逻辑。
assert discarded["extra_prefill_tokens"] > resumed["extra_prefill_tokens"]  # 执行本行的状态、计算或校验逻辑。


## 5. 版本失配：宁可重算也不能错误复用

工具返回可能附着在错误版本上，例如用户在等待期间编辑了问题，或模型的 context 被截断。这里强制拒绝；调用方可选择重新 prefill。这个失败分支是 cache correctness 的核心，而不是性能优化细节。


In [ ]:
old_snapshot = {"version": 0, "kv": ("kv-0", "kv-1"), "kind": "tool"}  # 执行本行的状态、计算或校验逻辑。
pause(session, "human")  # 执行本行的状态、计算或校验逻辑。
try:  # 执行本行的状态、计算或校验逻辑。
    resume(session, old_snapshot, [100])  # 执行本行的状态、计算或校验逻辑。
    assert False  # 执行本行的状态、计算或校验逻辑。
except ValueError:  # 执行本行的状态、计算或校验逻辑。
    assert session.phase == "paused"  # 执行本行的状态、计算或校验逻辑。
assert session.intercept_kind == "human"  # 执行本行的状态、计算或校验逻辑。


## 6. 内存预算：优先保留近期且快恢复的会话

KV 不能无限保留。下面的简化策略按预期恢复时间和 block 数排序，直到预算耗尽；生产中还需结合优先级、租户公平、swap 成本、fragmentation 与可预测的尾延迟。


In [ ]:
def admit_paused(candidates, block_budget):  # 执行本行的状态、计算或校验逻辑。
    chosen = []  # 执行本行的状态、计算或校验逻辑。
    used = 0  # 执行本行的状态、计算或校验逻辑。
    for item in sorted(candidates, key=lambda value: value["wait_ms"]):  # 执行本行的状态、计算或校验逻辑。
        if used + item["blocks"] <= block_budget:  # 执行本行的状态、计算或校验逻辑。
            chosen.append(item["id"])  # 执行本行的状态、计算或校验逻辑。
            used += item["blocks"]  # 执行本行的状态、计算或校验逻辑。
    return chosen, used  # 执行本行的状态、计算或校验逻辑。
candidates = [{"id": "fast", "wait_ms": 10, "blocks": 2}, {"id": "slow", "wait_ms": 5000, "blocks": 3}]  # 执行本行的状态、计算或校验逻辑。
chosen, used = admit_paused(candidates, 3)  # 执行本行的状态、计算或校验逻辑。
assert chosen == ["fast"]  # 执行本行的状态、计算或校验逻辑。
assert used == 2  # 执行本行的状态、计算或校验逻辑。
assert "slow" not in chosen  # 执行本行的状态、计算或校验逻辑。


## 7. 正确性指标：复用率和错误复用率要分开

只报告 KV hit rate 会掩盖风险：错误复用一次就可能污染答案。至少并列报告保存的 prefill token、恢复成功率、主动重算次数、内存占用和 snapshot mismatch；安全性指标应优先于吞吐指标。


In [ ]:
def cache_metrics(events):  # 执行本行的状态、计算或校验逻辑。
    saved = sum(event["saved"] for event in events)  # 执行本行的状态、计算或校验逻辑。
    reused = sum(event["reused"] for event in events)  # 执行本行的状态、计算或校验逻辑。
    mismatches = sum(event["mismatch"] for event in events)  # 执行本行的状态、计算或校验逻辑。
    return {"saved": saved, "reuse_rate": reused / len(events), "mismatch": mismatches}  # 执行本行的状态、计算或校验逻辑。
metrics = cache_metrics([{"saved": 4, "reused": 1, "mismatch": 0}, {"saved": 0, "reused": 0, "mismatch": 1}])  # 执行本行的状态、计算或校验逻辑。
assert metrics["saved"] == 4  # 执行本行的状态、计算或校验逻辑。
assert metrics["reuse_rate"] == 0.5  # 执行本行的状态、计算或校验逻辑。
assert metrics["mismatch"] == 1  # 执行本行的状态、计算或校验逻辑。


## 8. 制品：会话、KV、工具结果三者必须同版本

恢复制品最少应绑定 session id、model/chat-template 版本、tokenizer、KV 格式和工具调用 id。这里用摘要演示可追踪性；生产场景还要保护其中的用户内容和工具凭据。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
artifact = {"session": session.session_id, "model": "demo-v1", "kv_format": "paged-v1", "tool_call": "call-7"}  # 执行本行的状态、计算或校验逻辑。
fingerprint = hashlib.sha256(json.dumps(artifact, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert artifact["session"] == "s-1"  # 执行本行的状态、计算或校验逻辑。
assert artifact["kv_format"] == "paged-v1"  # 执行本行的状态、计算或校验逻辑。
assert len(fingerprint) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

按“任务目标 → 显式状态 → 动作前策略门禁 → 成功 oracle → 失败和重试 → 指标与版本化制品”的顺序回答。不要把一次文本看起来合理的演示当成可靠性证明：要独立检查状态、权限、不可逆副作用与多次运行的一致性。
